# contiguous-layout — ex4: predict which slice operations break contiguity

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `contiguous-layout`. Running the final beacon cell reports progress against the `PyTorch: Contiguous layout` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Contiguous layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`contiguous-layout`** (exercise 4). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "contiguous-layout"
DD_SUBTOPIC = "PyTorch: Contiguous layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Contiguous layout — quick refresher

A tensor is contiguous when its strides match the row-major formula `stride[i] = prod(shape[i+1:])`. Operations like `.transpose()`, `.permute()`, and `.t()` produce *views* with rearranged strides — they are NOT contiguous, so `view()` will fail. Slicing along the outer dim keeps contiguity; slicing/striding an inner dim does not.

### Exercise 4 — predict which slice operations break contiguity

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Evaluate
> LO: Evaluate a list of slice expressions on a contiguous base tensor and predict which ones return a still-contiguous view (outer-axis slices, full-axis slices) vs which break it (strided inner-axis slices, transpose-then-slice).
> Keywords: slicing, view-breakage, contig-prediction, outer-vs-inner-axis
> ```

**KCs targeted:** `contiguous-stride-formula`, `is-contiguous-check`

ex1 derived strides of contiguous tensors. ex3 classified contiguity from a `(shape, stride)` tuple. This drill *evaluates* a concrete slice expression and predicts its contiguity *without* executing it on a real tensor — the test then runs the slice and confirms.

Implement `ex4_predict_contiguous(shape, slice_spec)`:

1. `shape` is a tuple of ints — the base tensor is a contiguous `t.arange(prod(shape)).reshape(shape)`.
2. `slice_spec` is a tuple of Python `slice` objects, one per dim. Each is `slice(start, stop, step)` where any field may be `None`.
3. Predict whether the resulting view is contiguous and return `True` or `False`.

Rules to encode:
- If the slice on dim `i` has `step != 1`, contiguity breaks (unless it shrinks the dim to length 1 or 0).
- All inner dims (dim > leading-trimmed prefix) must be full-length slices (`slice(None, None, None)`).
- The leading prefix may have arbitrary `start:stop` with `step=1`, since outer-axis slicing keeps contig.

Edge cases: any axis sliced to length 0 or 1 effectively removes its contribution — `t.is_contiguous()` returns True. You may simply trust `t.is_contiguous()` on the concrete tensor as the ground truth for those edge cases (see the solution).

In [ ]:
def ex4_predict_contiguous(shape: tuple, slice_spec: tuple) -> bool:
    """Return True iff applying slice_spec to a contiguous arange(*shape) yields a contiguous view."""
    raise NotImplementedError()


def _test_ex4():
    import math

    def _ground_truth(shape, spec):
        base = t.arange(math.prod(shape)).reshape(shape)
        return base[spec].is_contiguous()

    CASES = [
        # (shape, slice_spec, label)
        ((4, 5),    (slice(0, 2),         slice(None)),       'outer prefix → contig'),
        ((4, 5),    (slice(None),         slice(0, 3)),       'inner partial → NOT contig'),
        ((4, 5),    (slice(None),         slice(None, None, 2)),  'inner step!=1 → NOT contig'),
        ((4, 5),    (slice(None, None, 2),slice(None)),       'outer step!=1 → NOT contig'),
        ((3, 4, 5), (slice(1, 3),         slice(None),        slice(None)), 'leading partial → contig'),
        ((3, 4, 5), (slice(None),         slice(0, 2),        slice(None)), 'middle partial → NOT contig'),
        ((3, 4, 5), (slice(None),         slice(None),        slice(0, 4)), 'inner partial → NOT contig'),
        ((3, 4, 5), (slice(None),         slice(None),        slice(None)), 'full slice → contig'),
        ((3, 4, 5), (slice(0, 1),         slice(None),        slice(None)), 'leading len-1 → contig'),
    ]
    for shape, spec, label in CASES:
        pred = ex4_predict_contiguous(shape, spec)
        truth = _ground_truth(shape, spec)
        assert pred == truth, f'mismatch on `{label}`: predicted {pred}, truth {truth}'

    # Property fuzz on a 4-D shape.
    rng = t.Generator().manual_seed(1)
    shape4 = (2, 3, 4, 5)
    for _ in range(30):
        spec = []
        for d in shape4:
            choice = t.randint(0, 4, (1,), generator=rng).item()
            if choice == 0:
                spec.append(slice(None))
            elif choice == 1:
                spec.append(slice(None, None, 2))
            elif choice == 2:
                spec.append(slice(0, max(d - 1, 1)))
            else:
                spec.append(slice(1, d))
        spec = tuple(spec)
        assert ex4_predict_contiguous(shape4, spec) == _ground_truth(shape4, spec), (
            f'fuzz fail on spec={spec}'
        )

    # --- Visualization: small grid of contig vs non-contig slices ---
    grid_specs = [
        (slice(None), slice(None)),
        (slice(0, 2), slice(None)),
        (slice(None), slice(0, 3)),
        (slice(None, None, 2), slice(None)),
    ]
    preds = [ex4_predict_contiguous((4, 5), s) for s in grid_specs]
    fig, axes = plt.subplots(1, len(grid_specs), figsize=(11, 2.5))
    base = t.arange(20).reshape(4, 5)
    for ax, spec, p in zip(axes, grid_specs, preds):
        sliced = base[spec]
        ax.imshow(sliced.numpy(), cmap='Blues')
        edge = 'green' if p else 'red'
        for s in ax.spines.values():
            s.set_edgecolor(edge); s.set_linewidth(3)
        ax.set_title(f'{spec}\n→ contig={p}')
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_predict_contiguous(shape: tuple, slice_spec: tuple) -> bool:
    # Row-major contig invariant after slicing:
    #   For each dim i with result-length > 1, EVERY dim j > i must be
    #   full-extent with step 1 (or trivial). Step != 1 on any length>1 dim
    #   also breaks contig.
    info = []
    for d, s in zip(shape, slice_spec):
        start, stop, step = s.indices(d)
        if step > 0:
            length = max(0, (stop - start + step - 1) // step)
        else:
            length = max(0, (start - stop - step - 1) // (-step))
        full = (start == 0 and stop == d and step == 1)
        info.append((length, full, step))
    # Step != 1 on a non-trivial dim → never contig.
    for length, full, step in info:
        if step != 1 and length > 1:
            return False
    # Collect the indices of non-trivial, non-full dims.
    nontriv_nonfull = [i for i, (length, full, _) in enumerate(info)
                       if length > 1 and not full]
    if len(nontriv_nonfull) > 1:
        return False
    if len(nontriv_nonfull) == 1:
        k = nontriv_nonfull[0]
        for j in range(k):
            # Any earlier dim with length > 1 (full or not) wrecks contig:
            # its stride won't match the shrunken inner product.
            if info[j][0] > 1:
                return False
    return True
```

**Outer partial = OK, inner partial = breaks.** Row-major storage means slicing the leading axis is just a pointer bump — the underlying bytes for the chosen rows are still contiguous. Slicing an INNER axis leaves gaps in the underlying buffer (you skip elements that belong to the rows you didn't drop), so strides no longer match the row-major formula.

**`step != 1` always breaks** (unless the resulting axis has length ≤ 1, in which case the dim is degenerate and PyTorch reports it as contiguous trivially).

**Why this is a one-line lookup in real code:** `tensor.is_contiguous()`. But predicting it from the slice spec alone — without instantiating anything — is the skill you need for **shape-checking decorators**, torch.fx tracing, and writing fused kernels.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex4',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()